# 06 — PSO Feature Selection (+ GA vs PSO)

From-scratch **binary PSO**. Continuous particle positions are mapped to binary masks with a **sigmoid transfer function**.

### Swarm narrative
- **Cognitive component (c1):** pull toward each particle's personal best mask.
- **Social component (c2):** pull toward the swarm's global best.
- **Inertia (w):** balances exploration (high w) and exploitation (low w).

In [1]:
from pathlib import Path
import random

import numpy as np

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [cwd, *cwd.parents] if (p / "environment.yml").exists()),
    cwd,
)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"

for d in (DATA_INTERIM, DATA_PROCESSED, FIGURES_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Random seed  : {RANDOM_SEED}")

Project root : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction
Random seed  : 42


In [2]:
import json
import time
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

X_train = pd.read_csv(DATA_PROCESSED / "X_train.csv")
y_train = pd.read_csv(DATA_PROCESSED / "y_train.csv").squeeze().values
feature_names = np.array(X_train.columns)
X = X_train.values
n_features = X.shape[1]

ga = json.loads((DATA_PROCESSED / "ga_feature_selection.json").read_text(encoding="utf-8"))
print("Loaded GA result:", ga["n_selected"], "features, cv_f1", ga["cv_f1_selected"])

Loaded GA result: 17 features, cv_f1 0.6886498781824854


In [3]:
# Match GA experimental budget approximately
SWARM_SIZE = 20
N_ITERS = 25
W = 0.7
C1 = 1.5
C2 = 1.5
LAMBDA_SPARSITY = 0.05
CV_SPLITS = 3
V_MAX = 4.0

rng = np.random.default_rng(RANDOM_SEED)

def make_rf():
    return RandomForestClassifier(
        n_estimators=60, max_depth=6, min_samples_leaf=5,
        class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1,
    )

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -10, 10)))

def position_to_mask(pos):
    # Binary PSO transfer: P(bit=1) = sigmoid(pos)
    probs = sigmoid(pos)
    mask = (rng.random(n_features) < probs).astype(int)
    if mask.sum() == 0:
        mask[int(np.argmax(probs))] = 1
    return mask

def fitness(mask: np.ndarray) -> float:
    mask = mask.astype(bool)
    if mask.sum() == 0:
        return 0.0
    X_sub = X[:, mask]
    cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_SEED)
    f1 = float(cross_val_score(make_rf(), X_sub, y_train, cv=cv, scoring="f1").mean())
    penalty = LAMBDA_SPARSITY * (mask.sum() / n_features)
    return f1 - penalty

In [4]:
t0 = time.perf_counter()
# Initialise positions in [-1, 1], velocities small
positions = rng.uniform(-1, 1, size=(SWARM_SIZE, n_features))
velocities = rng.uniform(-0.5, 0.5, size=(SWARM_SIZE, n_features))
masks = np.array([position_to_mask(p) for p in positions])
pbest_pos = positions.copy()
pbest_mask = masks.copy()
pbest_fit = np.array([fitness(m) for m in masks])

gbest_idx = int(np.argmax(pbest_fit))
gbest_pos = pbest_pos[gbest_idx].copy()
gbest_mask = pbest_mask[gbest_idx].copy()
gbest_fit = float(pbest_fit[gbest_idx])

best_hist = [gbest_fit]
mean_hist = [float(pbest_fit.mean())]

for it in range(N_ITERS):
    r1 = rng.random((SWARM_SIZE, n_features))
    r2 = rng.random((SWARM_SIZE, n_features))
    velocities = (
        W * velocities
        + C1 * r1 * (pbest_pos - positions)
        + C2 * r2 * (gbest_pos - positions)
    )
    velocities = np.clip(velocities, -V_MAX, V_MAX)
    positions = positions + velocities

    for i in range(SWARM_SIZE):
        mask = position_to_mask(positions[i])
        fit = fitness(mask)
        if fit > pbest_fit[i]:
            pbest_fit[i] = fit
            pbest_pos[i] = positions[i].copy()
            pbest_mask[i] = mask
        if fit > gbest_fit:
            gbest_fit = fit
            gbest_pos = positions[i].copy()
            gbest_mask = mask.copy()

    best_hist.append(gbest_fit)
    mean_hist.append(float(pbest_fit.mean()))
    print(f"Iter {it:02d} | gbest={gbest_fit:.4f} mean_pbest={pbest_fit.mean():.4f} bits={gbest_mask.sum()}")

pso_time = time.perf_counter() - t0
print(f"PSO finished in {pso_time:.1f}s")

Iter 00 | gbest=0.6388 mean_pbest=0.5896 bits=20


Iter 01 | gbest=0.6388 mean_pbest=0.5941 bits=20


Iter 02 | gbest=0.6489 mean_pbest=0.6046 bits=18


Iter 03 | gbest=0.6489 mean_pbest=0.6102 bits=18


Iter 04 | gbest=0.6489 mean_pbest=0.6135 bits=18


...

Iter 20 | gbest=0.6489 mean_pbest=0.6313 bits=18


Iter 21 | gbest=0.6489 mean_pbest=0.6313 bits=18


Iter 22 | gbest=0.6489 mean_pbest=0.6313 bits=18


Iter 23 | gbest=0.6489 mean_pbest=0.6313 bits=18


Iter 24 | gbest=0.6489 mean_pbest=0.6313 bits=18
PSO finished in 199.8s


In [5]:
selected_features = feature_names[gbest_mask.astype(bool)].tolist()
cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_SEED)
f1_full = float(cross_val_score(make_rf(), X, y_train, cv=cv, scoring="f1").mean())
f1_pso = float(cross_val_score(make_rf(), X[:, gbest_mask.astype(bool)], y_train, cv=cv, scoring="f1").mean())

pso_result = {
    "algorithm": "PSO",
    "n_total_features": int(n_features),
    "n_selected": int(gbest_mask.sum()),
    "reduction_pct": float(100 * (1 - gbest_mask.sum() / n_features)),
    "best_fitness": gbest_fit,
    "cv_f1_full": f1_full,
    "cv_f1_selected": f1_pso,
    "runtime_sec": pso_time,
    "selected_features": selected_features,
    "mask": gbest_mask.astype(int).tolist(),
}
(DATA_PROCESSED / "pso_feature_selection.json").write_text(json.dumps(pso_result, indent=2), encoding="utf-8")
pd.Series(selected_features).to_csv(DATA_PROCESSED / "pso_selected_features.csv", index=False, header=["feature"])

# Jaccard overlap with GA
ga_set = set(ga["selected_features"])
pso_set = set(selected_features)
jaccard = len(ga_set & pso_set) / len(ga_set | pso_set) if (ga_set | pso_set) else 0.0

# Winner: higher CV F1; tie-break fewer features
ga_f1, pso_f1 = ga["cv_f1_selected"], f1_pso
if pso_f1 > ga_f1 or (np.isclose(pso_f1, ga_f1) and pso_result["n_selected"] < ga["n_selected"]):
    winner = "PSO"
    winning = pso_result
else:
    winner = "GA"
    winning = ga

comparison = {
    "ga_cv_f1": ga_f1,
    "pso_cv_f1": pso_f1,
    "ga_n_selected": ga["n_selected"],
    "pso_n_selected": pso_result["n_selected"],
    "ga_runtime_sec": ga["runtime_sec"],
    "pso_runtime_sec": pso_time,
    "jaccard_overlap": jaccard,
    "winner": winner,
    "winning_features": winning["selected_features"],
    "winning_mask": winning["mask"],
}
(DATA_PROCESSED / "nia_feature_selection_winner.json").write_text(json.dumps(comparison, indent=2), encoding="utf-8")
pd.Series(winning["selected_features"]).to_csv(
    DATA_PROCESSED / "winning_selected_features.csv", index=False, header=["feature"]
)

print(json.dumps({k: v for k, v in comparison.items() if k not in ("winning_features", "winning_mask")}, indent=2))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(best_hist, label="gbest")
ax.plot(mean_hist, label="mean pbest")
ax.set_xlabel("Iteration")
ax.set_ylabel("Fitness")
ax.set_title("PSO fitness history")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "06_pso_fitness_history.png", dpi=150)
plt.show()

# Comparison bar
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Full", "GA", "PSO"], [f1_full, ga_f1, pso_f1], color=["#8d99ae", "#2a9d8f", "#e9c46a"])
ax.set_ylabel("CV F1")
ax.set_title("Feature subsets vs full set (RF CV F1)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "06_ga_vs_pso_f1.png", dpi=150)
plt.show()

{
  "ga_cv_f1": 0.6886498781824854,
  "pso_cv_f1": 0.6761879740677464,
  "ga_n_selected": 17,
  "pso_n_selected": 18,
  "ga_runtime_sec": 236.8041946000012,
  "pso_runtime_sec": 199.80980809999892,
  "jaccard_overlap": 0.5217391304347826,
  "winner": "GA"
}


C:\Users\ishan\AppData\Local\Temp\ipykernel_24424\4175909496.py:63: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\ishan\AppData\Local\Temp\ipykernel_24424\4175909496.py:72: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# Visuals: feature counts + overlap (from saved GA/PSO results)
import json
import matplotlib.pyplot as plt

ga = json.loads((DATA_PROCESSED / "ga_feature_selection.json").read_text(encoding="utf-8"))
pso_result = json.loads((DATA_PROCESSED / "pso_feature_selection.json").read_text(encoding="utf-8"))
comparison = json.loads((DATA_PROCESSED / "nia_feature_selection_winner.json").read_text(encoding="utf-8"))
n_features = ga["n_total_features"]
ga_set = set(ga["selected_features"])
pso_set = set(pso_result["selected_features"])
jaccard = comparison["jaccard_overlap"]
winner = comparison["winner"]

fig, ax = plt.subplots(figsize=(6, 4))
ns = [n_features, ga["n_selected"], pso_result["n_selected"]]
ax.bar(["Full", "GA", "PSO"], ns, color=["#8d99ae", "#2a9d8f", "#e9c46a"])
ax.set_ylabel("Number of features")
ax.set_title("Feature count: Full vs GA vs PSO")
for i, v in enumerate(ns):
    ax.text(i, v + 0.2, str(int(v)), ha="center")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "06_feature_counts.png", dpi=150)
plt.show()

only_ga = len(ga_set - pso_set)
only_pso = len(pso_set - ga_set)
both = len(ga_set & pso_set)
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["GA only", "Shared", "PSO only"], [only_ga, both, only_pso], color=["#2a9d8f", "#264653", "#e9c46a"])
ax.set_ylabel("Feature count")
ax.set_title(f"GA vs PSO overlap (Jaccard={jaccard:.2f})")
for i, v in enumerate([only_ga, both, only_pso]):
    ax.text(i, v + 0.1, str(v), ha="center")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "06_ga_pso_overlap.png", dpi=150)
plt.show()
print("Winner:", winner)


**Insight (simple):** **Shared** features are stable signals both algorithms like. The **winner** is whoever has higher CV F1 (tie → fewer features). TabNet will train on that winning list.

**Decision rule:** Winner = higher stratified CV F1; if tied, prefer the smaller subset. TabNet in notebook `07` trains primarily on this winning mask.

**Next:** Notebook `07` — TabNet on the winning subset (+ short full-feature before/after).